# Salary Regression

In [21]:
import datetime
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split as TTS
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

In [2]:
data = pd.read_csv('./data/churn_modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
# Delete some columns
data.drop(['RowNumber','CustomerId','Surname'], axis=1, inplace=True)

In [ ]:
# Encode 'Gender' column
gender_encoder = LabelEncoder()
data['Gender'] = gender_encoder.fit_transform(data['Gender'])

In [9]:
# Onehot encode for 'Geography' column
geo_encoder = OneHotEncoder(sparse_output=False)
geo_encoded_df = pd.DataFrame(geo_encoder.fit_transform(data[['Geography']]), 
                              columns=geo_encoder.get_feature_names_out())
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [10]:
# Combine onehot encoded data into original data
data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [11]:
# Separate data into dependent and independents
X = data.drop('EstimatedSalary', axis=1)
y = data['EstimatedSalary']

In [12]:
X_train, X_test, y_train, y_test = TTS(X, y, test_size=0.2, random_state=42)
print(X_train.shape)

# Standard scaling to the independent data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

(8000, 12)


In [36]:
# Save encoders and scaler
with open('./preprocessors/regression/gender_label_encoder.pkl', 'wb') as file:
    pickle.dump(gender_encoder, file)

with open('./preprocessors/regression/onehot_geo_encoder.pkl', 'wb') as file:
    pickle.dump(geo_encoder, file)

with open('./preprocessors/regression/standard_scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

### ANN Regression Problem Statement

In [15]:
model = Sequential([Dense(64, activation='relu', input_shape=(X_train.shape[1],)), # HL1 connected with Input Layer
                    Dense(32, activation='relu'),   # HL2
                    Dense(1)])                      # Output Layer with no activation (linear)

/home/htet-aung-lynn/Study/E2E-Churn-Prediction-with-ANN/venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1789992270.197406  472532 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [19]:
# Complie model
model.compile(optimizer = "adam",          
              loss = "mean_absolute_error",   
              metrics = ["mae"])            # Get as a list

In [20]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [22]:
# Set up Tensorboard
log_dir = "logs/regression/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [24]:
# Set up Early Stopping
early_stopping_callback = EarlyStopping(monitor = 'val_loss', 
                                        patience = 10,  # Allow up to 10 consecutive epochs without improvement before stopping training
                                        restore_best_weights = True) # restore the best weight after monitoring next 10 epochs

In [29]:
history = model.fit(X_train_scaled,
                    y_train,
                    validation_data = (X_test, y_test),
                    epochs = 100,
                    callbacks = [tensorboard_callback, early_stopping_callback])

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 100224.3438 - mae: 100224.3438 - val_loss: 4442021.5000 - val_mae: 4442021.5000
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 100200.1172 - mae: 100200.1172 - val_loss: 4965922.0000 - val_mae: 4965922.0000
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 100169.0859 - mae: 100169.0859 - val_loss: 5654821.5000 - val_mae: 5654821.5000
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 100127.9375 - mae: 100127.9375 - val_loss: 6590169.0000 - val_mae: 6590169.0000
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 100071.7422 - mae: 100071.7422 - val_loss: 7891723.0000 - val_mae: 7891723.0000
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 99991.9375 - mae: 99991.9375 - val_loss: 9784447.0000 - val_mae: 9784447.0000
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 99873.1875 - mae: 99873.1875 - val_loss: 12734681.0000 - val_mae: 12734681.0000
Epoch 8/100
250

In [30]:
#  Load Tensorboard extension
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [31]:
%tensorboard --logdir logs/regression/

Reusing TensorBoard on port 6008 (pid 490192), started 0:00:23 ago. (Use '!kill 490192' to kill it.)

In [32]:
# Evaluate model on the test data
test_loss, test_mae = model.evaluate(X_test, y_test)
print(f"Test MAE: {test_mae}")

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 943us/step - loss: 4442021.5000 - mae: 4442021.5000
Test MAE: 4442021.5


In [33]:
# Save model.h5 -> h5 is compatible for Keras
model.save('./dl_model/regression_model.h5')

In [34]:
X_train.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,Exited,Geography_France,Geography_Germany,Geography_Spain
9254,686,1,32,6,0.00,2,1,1,0,1.0,0.0,0.0
1561,632,1,42,4,119624.60,2,1,1,0,0.0,1.0,0.0
1670,559,1,24,3,114739.92,1,1,0,1,0.0,0.0,1.0
6087,561,0,27,9,135637.00,1,1,0,1,1.0,0.0,0.0
6669,517,1,56,9,142147.32,1,0,0,1,1.0,0.0,0.0
